In [24]:
import os
import re
import csv
import pandas as pd
from numpy import trapezoid


def parse_graph_stats(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    name_match = re.search(r'графе (.+?):', content)
    graph_name = name_match.group(1).strip() if name_match else os.path.basename(filepath)

    vertices = int(re.search(r'Количество вершин: (\d+)', content).group(1))
    edges = int(re.search(r'Количество рёбер: (\d+)', content).group(1))
    density = float(re.search(r'Плотность графа: ([\d.]+)', content).group(1))
    count_of_wcc = int(re.search(r'Количество WCC: (\d+)', content).group(1))
    v_largest_wcc = float(re.search(r'Доля вершин в наибольшей WCC: ([\d.]+)', content).group(1))
    count_of_scc = int(re.search(r'Количество SCC: (\d+)', content).group(1))
    v_largest_scc = float(re.search(r'Доля вершин в наибольшей SCC: ([\d.]+)', content).group(1))
    diameter_ds = int(re.search(r'Диаметр наибольшей WCC \(The Double Sweep\): (\d+)', content).group(1))
    percentile_ds = float(re.search(r'90 процентиль расстояний: ([\d.]+)', content).group(1))
    diameter_s = int(re.search(r'Диаметр наибольшей WCC \(Snowball \+ Double Sweep\): (\d+)', content).group(1))
    percentile_s = float(re.search(r'90 процентиль расстояний \(Snowball\): ([\d.]+)', content).group(1))
    triangles = int(re.search(r'Количество треугольников: (\d+)', content).group(1))
    avg_clustering = float(re.search(r'Средний коэффициент кластеризации: ([\d.]+)', content).group(1))
    global_clustering = float(re.search(r'Глобальный коэффициент кластеризации: ([\d.]+)', content).group(1))
    avg_clustering_wcc = float(re.search(r'Средний коэффициент кластеризации \(largest WCC\): ([\d.]+)', content).group(1))
    min_deg = float(re.search(r'Минимальная степень узлов: (\d+)', content).group(1))
    avg_deg = float(re.search(r'Средняя степень узлов: ([\d.]+)', content).group(1))
    max_deg = float(re.search(r'Максимальная степень узлов: (\d+)', content).group(1))

    def extract_block(header):
        match = re.search(header + r':\s*((?:\n\t+x = [^\n]+)+)', content)
        return match.group(1) if match else ''

    def extract_removal_data(block):
        data = {}
        for match in re.findall(r'x = ([\d.]+)%: доля вершин в наибольшей WCC: ([\d.]+)', block):
            x = float(match[0])
            y = float(match[1])
            data[x] = y
        return data

    block_random = extract_block(r'Удаление случайных узлов')
    block_degree = extract_block(r'Удаление узлов наибольшей степени')

    data_random = extract_removal_data(block_random)
    data_degree = extract_removal_data(block_degree)

    def compute_auc(data):
        if not data: return None
        x = sorted(data.keys())
        y = [data[k] for k in x]
        return trapezoid(y, x)

    auc_random = compute_auc(data_random)
    auc_degree = compute_auc(data_degree)

    return {
        'graph': graph_name,
        'vertices': vertices,
        'edges': edges,
        'density': density,
        'count_of_wcc': count_of_wcc,
        'v_largest_wcc': v_largest_wcc,
        'count_of_scc': count_of_scc,
        'v_largest_scc': v_largest_scc,
        'diameter_ds': diameter_ds,
        'percentile_ds': percentile_ds,
        'diameter_s': diameter_s,
        'percentile_s': percentile_s,
        'triangles': triangles,
        'avg_clustering': avg_clustering,
        'global_clustering': global_clustering,
        'avg_clustering_wcc': avg_clustering_wcc,
        'min_deg': min_deg,
        'avg_deg': avg_deg,
        'max_deg': max_deg,
        'auc_random': auc_random,
        'auc_degree': auc_degree
    }



input_dir = r'C:\Users\Dmitry\Desktop\Graph_Theory_LeetCode25\src\output'
output_csv = os.path.join(r'C:\Users\Dmitry\Desktop\Graph_Theory_LeetCode25\src\visualization', 'graph_resilience.csv')

results = []

for file in os.listdir(input_dir):
    if file.endswith('.txt'):
        full_path = os.path.join(input_dir, file)
        stats = parse_graph_stats(full_path)
        results.append(stats)


with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=[
        'graph', 'vertices', 'edges', 'density', 'count_of_wcc', 'v_largest_wcc', 
        'count_of_scc', 'v_largest_scc', 'diameter_ds', 'percentile_ds', 'diameter_s',
        'percentile_s', 'triangles', 'avg_clustering', 'global_clustering', 'avg_clustering_wcc',
        'min_deg', 'avg_deg', 'max_deg', 'auc_random', 'auc_degree'
    ])
    writer.writeheader()
    for row in results:
        writer.writerow(row)

print("CSV сохранён:", output_csv)

df = pd.read_csv(output_csv)
display(df)


CSV сохранён: C:\Users\Dmitry\Desktop\Graph_Theory_LeetCode25\src\visualization\graph_resilience.csv


,graph,vertices,edges,density,count_of_wcc,v_largest_wcc,count_of_scc,v_largest_scc,diameter_ds,percentile_ds,...,percentile_s,triangles,avg_clustering,global_clustering,avg_clustering_wcc,min_deg,avg_deg,max_deg,auc_random,auc_degree
0,CA-AstroPh,18772,396160,0.001124,290,0.953708,290,0.953708,14,6.0,...,5.0,1351441,0.6306,0.3179,0.6328,0.0,21.10,504.0,64.16075,21.497750
1,ca-coauthors-dblp,540486,15245729,0.000104,1,1.000000,1,1.000000,23,7.0,...,5.0,444095058,0.8019,0.6562,0.8019,1.0,56.41,3299.0,71.48700,46.063000
2,CA-GrQc,5242,28980,0.001055,355,0.793209,355,0.793209,17,8.0,...,7.0,48260,0.5296,0.6297,0.5569,0.0,5.53,81.0,35.76425,2.038000
3,com-orkut.ungraph,3072441,117185083,0.000025,1,1.000000,1,1.000000,9,5.0,...,3.0,627584181,0.1666,0.0413,0.1666,1.0,76.28,33313.0,77.68450,54.879250
4,com-youtube.ungraph,1134890,2987624,0.000005,1,1.000000,1,1.000000,24,7.0,...,4.0,3056386,0.0808,0.0062,0.0808,1.0,5.27,28754.0,45.92450,3.143976
5,Email-EuAll,265214,420045,0.000010,15836,0.847738,231000,0.128964,14,5.0,...,2.0,267313,0.0671,0.0041,0.0791,0.0,2.75,7636.0,7.14980,0.090664
6,musae_git_edges,37700,289003,0.000407,1,1.000000,37700,0.000027,11,4.0,...,2.0,523810,0.1675,0.0124,0.1675,1.0,15.33,9458.0,66.19150,9.118000
7,soc-wiki-Vote,889,2914,0.007383,1,1.000000,889,0.001125,13,6.0,...,5.0,2119,0.1528,0.1273,0.1528,1.0,6.56,102.0,55.87100,11.151250
8,vk,3215720,17414510,0.000003,24337,0.983362,3215720,0.000000,19,7.0,...,6.0,108030337,0.0494,0.1095,0.0499,1.0,10.83,6503.0,60.08175,12.474500
9,web-Google,875713,5105039,0.000011,2746,0.977263,371764,0.496530,24,9.0,...,5.0,13391903,0.5143,0.0552,0.5190,1.0,9.87,6332.0,53.05725,5.091250


**На основе таблицы можно сделать следующий вывод:**

In [56]:
print(f"1. Наибольшее количество вершин {df['vertices'].max()} у графа {df[df['vertices'] == df['vertices'].max()].iloc[0]['graph']}")
print(f"2. Наименьшее количество вершин {df['vertices'].min()} у графа {df[df['vertices'] == df['vertices'].min()].iloc[0]['graph']}")
print(f"3. Наибольшее количество ребер {df['edges'].max()} у графа {df[df['edges'] == df['edges'].max()].iloc[0]['graph']}")
print(f"4. Наименьшее количество ребер {df['edges'].min()} у графа {df[df['edges'] == df['edges'].min()].iloc[0]['graph']}")
print(f"5. Самая высокая плотность {df['density'].max()} у графа {df[df['density'] == df['density'].max()].iloc[0]['graph']}")
print(f"6. Самая низкая плотность {df['density'].min()} у графа {df[df['density'] == df['density'].min()].iloc[0]['graph']}")
print(f"7. Наибольшее количество WCC {df['count_of_wcc'].max()} у графа {df[df['count_of_wcc'] == df['count_of_wcc'].max()].iloc[0]['graph']} с долей вершин {df[df['count_of_wcc'] == df['count_of_wcc'].max()].iloc[0]['v_largest_wcc']}")
print(f"8. Наименьшее количество WCC {df['count_of_wcc'].min()} у графа {df[df['count_of_wcc'] == df['count_of_wcc'].min()].iloc[0]['graph']} с долей вершин {df[df['count_of_wcc'] == df['count_of_wcc'].min()].iloc[0]['v_largest_wcc']}")
print(f"9. Наибольшее количество SCC {df['count_of_scc'].max()} у графа {df[df['count_of_scc'] == df['count_of_scc'].max()].iloc[0]['graph']} с долей вершин {df[df['count_of_scc'] == df['count_of_scc'].max()].iloc[0]['v_largest_scc']}")
print(f"10. Наименьшее количество SCC {df['count_of_scc'].min()} у графа {df[df['count_of_scc'] == df['count_of_scc'].min()].iloc[0]['graph']} с долей вершин {df[df['count_of_scc'] == df['count_of_scc'].min()].iloc[0]['v_largest_scc']}")
print(f"11. Самый большой диаметр WCC {df['diameter_ds'].max()} у графа {df[df['diameter_ds'] == df['diameter_ds'].max()].iloc[0]['graph']}")
print(f"12. Самый маленький диаметр WCC {df['diameter_ds'].min()} у графа {df[df['diameter_ds'] == df['diameter_ds'].min()].iloc[0]['graph']}")
print(f"13. Наибольшее количество треугольников {df['triangles'].max()} у графа {df[df['triangles'] == df['triangles'].max()].iloc[0]['graph']}")
print(f"14. Наименьшее количество треугольников {df['triangles'].min()} у графа {df[df['triangles'] == df['triangles'].min()].iloc[0]['graph']}")
print('15. Минимальная степень узлов равна 0, то есть графы могут содержать изолированные вершины.')
print('16. Столбцы auc_random и auc_degree показывают площадь под кривой при удалении случайных узлов и удалении узлов наибольшей степени. Чем выше это значение, тем устойчивее граф к данному типу удаления.')
print('17. Наиболее устойчивыми графами являются com-orkut.ungraph и ca-coauthors-dbip, а наименее - Email-EuAll.')
print('18. Можно заметить, что наиболее устойчивые графы имеют наивысшую среднюю степень узлов. С наименее устойчивыми ситуация та же.')


1. Наибольшее количество вершин 3215720 у графа vk
2. Наименьшее количество вершин 889 у графа soc-wiki-Vote
3. Наибольшее количество ребер 117185083 у графа com-orkut.ungraph
4. Наименьшее количество ребер 2914 у графа soc-wiki-Vote
5. Самая высокая плотность 0.007383 у графа soc-wiki-Vote
6. Самая низкая плотность 3e-06 у графа vk
7. Наибольшее количество WCC 24337 у графа vk с долей вершин 0.983362
8. Наименьшее количество WCC 1 у графа ca-coauthors-dblp с долей вершин 1.0
9. Наибольшее количество SCC 3215720 у графа vk с долей вершин 0.0
10. Наименьшее количество SCC 1 у графа ca-coauthors-dblp с долей вершин 1.0
11. Самый большой диаметр WCC 164 у графа web-Stanford
12. Самый маленький диаметр WCC 6 у графа Wiki-Vote
13. Наибольшее количество треугольников 627584181 у графа com-orkut.ungraph
14. Наименьшее количество треугольников 2119 у графа soc-wiki-Vote
15. Минимальная степень узлов равна 0, то есть графы могут содержать изолированные вершины.
16. Столбцы auc_random и auc_degr

In [81]:
import pandas as pd
import re
from collections import defaultdict

filepath = r'..\graph\landmarkAlgo\benchmark\output\soc-wiki-Vote-landmarkBasic-bench.txt'

with open(filepath, 'r') as f:
    lines = f.readlines()

results = defaultdict(dict)

current_algorithm = None
current_landmarks = None

for line in lines:
    line = line.strip()

    if line.startswith('graph_theory/graph/landmarkAlgo'):
        current_algorithm = line.split('.')[-1]

    elif line.startswith('landmarks:'):
        current_landmarks = int(re.search(r'\d+', line).group())

    elif current_algorithm and current_landmarks:
        if line.startswith('precomputing time:'):
            results[(current_landmarks, 'precomputing time (ms)')][current_algorithm] = int(re.search(r'\d+', line).group())
        elif line.startswith('Avg estimated dist:'):
            results[(current_landmarks, 'Avg estimated dist')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())
        elif line.startswith('Avg relative error:'):
            results[(current_landmarks, 'Avg relative error')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())
        elif line.startswith('Avg time algorithm:'):
            results[(current_landmarks, 'Avg time algorithm (ms)')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())
        elif line.startswith('file size:'):
            results[(current_landmarks, 'file size (MB)')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())

df = pd.DataFrame(results).T
df.index.names = ['landmarks', 'metric']
df = df.sort_index()

df

SelectRandomNodes  SelectHighestDegree  \
landmarks metric                                                            
4         Avg estimated dist                  5.0000               4.0000   
          Avg relative error                  0.6667               0.3333   
          Avg time algorithm (ms)             0.0400               0.0300   
          file size (MB)                      0.0000               0.0000   
          precomputing time (ms)              1.0000               1.0000   
8         Avg estimated dist                  6.0000               4.0000   
          Avg relative error                  1.0000               0.3333   
          Avg time algorithm (ms)             0.0600               0.0300   
          file size (MB)                      0.0100               0.0100   
          precomputing time (ms)              1.0000               1.0000   
16        Avg estimated dist                  4.0000               4.0000   
          Avg relative error                  0.3333               0.3333   
          Avg time algorithm (ms)             0.1000               0.0500   
          file size (MB)                      0.0100               0.0100   
          precomputing time (ms)              1.0000               2.0000   
32        Avg estimated dist                  3.0000               3.0000   
          Avg relative error                  0.0000               0.0000   
          Avg time algorithm (ms)             0.1800               0.0900   
          file size (MB)                      0.0300               0.0300   
          precomputing time (ms)              4.0000               3.0000   
64        Avg estimated dist                  4.0000               3.0000   
          Avg relative error                  0.3333               0.0000   
          Avg time algorithm (ms)             0.3500               0.1700   
          file size (MB)                      0.0500               0.0500   
          precomputing time (ms)              7.0000               4.0000   
128       Avg estimated dist                  3.0000               3.0000   
          Avg relative error                  0.0000               0.0000   
          Avg time algorithm (ms)             0.6800               0.3400   
          file size (MB)                      0.1100               0.1100   
          precomputing time (ms)             14.0000               7.0000   

                                   SelectBestCoverage  
landmarks metric                                       
4         Avg estimated dist                   4.0000  
          Avg relative error                   0.3333  
          Avg time algorithm (ms)              0.0200  
          file size (MB)                       0.0000  
          precomputing time (ms)              25.0000  
8         Avg estimated dist                   3.0000  
          Avg relative error                   0.0000  
          Avg time algorithm (ms)              0.0300  
          file size (MB)                       0.0100  
          precomputing time (ms)              20.0000  
16        Avg estimated dist                   3.0000  
          Avg relative error                   0.0000  
          Avg time algorithm (ms)              0.0500  
          file size (MB)                       0.0100  
          precomputing time (ms)              20.0000  
32        Avg estimated dist                   3.0000  
          Avg relative error                   0.0000  
          Avg time algorithm (ms)              0.0900  
          file size (MB)                       0.0300  
          precomputing time (ms)              22.0000  
64        Avg estimated dist                   3.0000  
          Avg relative error                   0.0000  
          Avg time algorithm (ms)              0.1800  
          file size (MB)                       0.0500  
          precomputing time (ms)              42.0000  
128       Avg estimated dist                   3.0000  
  

In [82]:
import pandas as pd
import re
from collections import defaultdict

filepath = r'..\graph\landmarkAlgo\benchmark\output\soc-wiki-Vote-landmarkShortcut-bench.txt'

with open(filepath, 'r') as f:
    lines = f.readlines()

results = defaultdict(dict)

current_algorithm = None
current_landmarks = None

for line in lines:
    line = line.strip()

    if line.startswith('graph_theory/graph/landmarkAlgo'):
        current_algorithm = line.split('.')[-1]

    elif line.startswith('landmarks:'):
        current_landmarks = int(re.search(r'\d+', line).group())

    elif current_algorithm and current_landmarks:
        if line.startswith('precomputing time:'):
            results[(current_landmarks, 'precomputing time (ms)')][current_algorithm] = int(re.search(r'\d+', line).group())
        elif line.startswith('Avg estimated dist:'):
            results[(current_landmarks, 'Avg estimated dist')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())
        elif line.startswith('Avg relative error:'):
            results[(current_landmarks, 'Avg relative error')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())
        elif line.startswith('Avg time algorithm:'):
            results[(current_landmarks, 'Avg time algorithm (ms)')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())
        elif line.startswith('file size:'):
            results[(current_landmarks, 'file size (MB)')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())

df = pd.DataFrame(results).T
df.index.names = ['landmarks', 'metric']
df = df.sort_index()

df

SelectRandomNodes  SelectHighestDegree  \
landmarks metric                                                            
4         Avg estimated dist                  4.0000               4.0000   
          Avg relative error                  0.3333               0.3333   
          Avg time algorithm (ms)             0.0500               0.0400   
          file size (MB)                      0.0100               0.0100   
          precomputing time (ms)              0.0000               1.0000   
8         Avg estimated dist                  4.0000               4.0000   
          Avg relative error                  0.3333               0.3333   
          Avg time algorithm (ms)             0.1100               0.0800   
          file size (MB)                      0.0300               0.0300   
          precomputing time (ms)              1.0000               1.0000   
16        Avg estimated dist                  4.0000               4.0000   
          Avg relative error                  0.3333               0.3333   
          Avg time algorithm (ms)             0.1900               0.1600   
          file size (MB)                      0.0500               0.0500   
          precomputing time (ms)              1.0000               2.0000   
32        Avg estimated dist                  3.0000               4.0000   
          Avg relative error                  0.0000               0.3333   
          Avg time algorithm (ms)             0.4000               0.3200   
          file size (MB)                      0.1100               0.1100   
          precomputing time (ms)              2.0000              13.0000   
64        Avg estimated dist                  3.0000               3.0000   
          Avg relative error                  0.0000               0.0000   
          Avg time algorithm (ms)             0.8200               0.6500   
          file size (MB)                      0.2200               0.2200   
          precomputing time (ms)              5.0000               5.0000   
128       Avg estimated dist                  3.0000               3.0000   
          Avg relative error                  0.0000               0.0000   
          Avg time algorithm (ms)             1.5900               1.3400   
          file size (MB)                      0.4300               0.4300   
          precomputing time (ms)              9.0000              10.0000   

                                   SelectBestCoverage  
landmarks metric                                       
4         Avg estimated dist                   4.0000  
          Avg relative error                   0.3333  
          Avg time algorithm (ms)              0.0400  
          file size (MB)                       0.0100  
          precomputing time (ms)              18.0000  
8         Avg estimated dist                   4.0000  
          Avg relative error                   0.3333  
          Avg time algorithm (ms)              0.0800  
          file size (MB)                       0.0300  
          precomputing time (ms)              20.0000  
16        Avg estimated dist                   4.0000  
          Avg relative error                   0.3333  
          Avg time algorithm (ms)              0.1700  
          file size (MB)                       0.0500  
          precomputing time (ms)              21.0000  
32        Avg estimated dist                   4.0000  
          Avg relative error                   0.3333  
          Avg time algorithm (ms)              0.3400  
          file size (MB)                       0.1100  
          precomputing time (ms)              22.0000  
64        Avg estimated dist                   3.0000  
          Avg relative error                   0.0000  
          Avg time algorithm (ms)              0.7000  
          file size (MB)                       0.2200  
          precomputing time (ms)              44.0000  
128       Avg estimated dist                   3.0000  
  

In [83]:
import pandas as pd
import re
from collections import defaultdict

filepath = r'..\graph\landmarkAlgo\benchmark\output\web-Google-landmarkBasic-bench.txt'

with open(filepath, 'r') as f:
    lines = f.readlines()

results = defaultdict(dict)

current_algorithm = None
current_landmarks = None

for line in lines:
    line = line.strip()

    if line.startswith('graph_theory/graph/landmarkAlgo'):
        current_algorithm = line.split('.')[-1]

    elif line.startswith('landmarks:'):
        current_landmarks = int(re.search(r'\d+', line).group())

    elif current_algorithm and current_landmarks:
        if line.startswith('precomputing time:'):
            results[(current_landmarks, 'precomputing time (ms)')][current_algorithm] = int(re.search(r'\d+', line).group())
        elif line.startswith('Avg estimated dist:'):
            results[(current_landmarks, 'Avg estimated dist')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())
        elif line.startswith('Avg relative error:'):
            results[(current_landmarks, 'Avg relative error')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())
        elif line.startswith('Avg time algorithm:'):
            results[(current_landmarks, 'Avg time algorithm (ms)')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())
        elif line.startswith('file size:'):
            results[(current_landmarks, 'file size (MB)')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())

df = pd.DataFrame(results).T
df.index.names = ['landmarks', 'metric']
df = df.sort_index()

df

SelectRandomNodes  SelectHighestDegree  \
landmarks metric                                                            
4         Avg estimated dist                 10.0000               8.0000   
          Avg relative error                  0.6667               0.3333   
          Avg time algorithm (ms)             0.0300               0.0400   
          file size (MB)                      3.3400               3.3400   
          precomputing time (ms)            738.0000            1154.0000   
8         Avg estimated dist                 10.0000               8.0000   
          Avg relative error                  0.6667               0.3333   
          Avg time algorithm (ms)             0.0500               0.0600   
          file size (MB)                      6.6800               6.6800   
          precomputing time (ms)           1034.0000            1443.0000   
16        Avg estimated dist                 10.0000               8.0000   
          Avg relative error                  0.6667               0.3333   
          Avg time algorithm (ms)             0.1300               0.1500   
          file size (MB)                     13.3600              13.3600   
          precomputing time (ms)           2016.0000            2411.0000   
32        Avg estimated dist                 10.0000               7.0000   
          Avg relative error                  0.6667               0.1667   
          Avg time algorithm (ms)             0.3500               0.4000   
          file size (MB)                     26.7200              26.7200   
          precomputing time (ms)           3467.0000            4004.0000   
64        Avg estimated dist                  9.0000               7.0000   
          Avg relative error                  0.5000               0.1667   
          Avg time algorithm (ms)             1.1900               1.3400   
          file size (MB)                     53.4500              53.4500   
          precomputing time (ms)           6955.0000            7465.0000   
128       Avg estimated dist                 10.0000               6.0000   
          Avg relative error                  0.6667               0.0000   
          Avg time algorithm (ms)             4.1100               4.6500   
          file size (MB)                    106.9000             106.9000   
          precomputing time (ms)          13238.0000           14546.0000   

                                   SelectBestCoverage  
landmarks metric                                       
4         Avg estimated dist                     6.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)                0.52  
          file size (MB)                         3.34  
          precomputing time (ms)             32189.00  
8         Avg estimated dist                     6.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)                0.83  
          file size (MB)                         6.68  
          precomputing time (ms)             32900.00  
16        Avg estimated dist                     6.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)                1.53  
          file size (MB)                        13.36  
          precomputing time (ms)             35612.00  
32        Avg estimated dist                     6.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)                2.96  
          file size (MB)                        26.72  
          precomputing time (ms)             37576.00  
64        Avg estimated dist                     6.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)                5.80  
          file size (MB)                        53.45  
          precomputing time (ms)             38555.00  
128       Avg estimated dist                     6.00  
  

In [84]:
import pandas as pd
import re
from collections import defaultdict

filepath = r'..\graph\landmarkAlgo\benchmark\output\web-Google-landmarkShortcut-bench.txt'

with open(filepath, 'r') as f:
    lines = f.readlines()

results = defaultdict(dict)

current_algorithm = None
current_landmarks = None

for line in lines:
    line = line.strip()

    if line.startswith('graph_theory/graph/landmarkAlgo'):
        current_algorithm = line.split('.')[-1]

    elif line.startswith('landmarks:'):
        current_landmarks = int(re.search(r'\d+', line).group())

    elif current_algorithm and current_landmarks:
        if line.startswith('precomputing time:'):
            results[(current_landmarks, 'precomputing time (ms)')][current_algorithm] = int(re.search(r'\d+', line).group())
        elif line.startswith('Avg estimated dist:'):
            results[(current_landmarks, 'Avg estimated dist')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())
        elif line.startswith('Avg relative error:'):
            results[(current_landmarks, 'Avg relative error')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())
        elif line.startswith('Avg time algorithm:'):
            results[(current_landmarks, 'Avg time algorithm (ms)')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())
        elif line.startswith('file size:'):
            results[(current_landmarks, 'file size (MB)')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())

df = pd.DataFrame(results).T
df.index.names = ['landmarks', 'metric']
df = df.sort_index()

df

SelectRandomNodes  SelectHighestDegree  \
landmarks metric                                                            
4         Avg estimated dist                    5.00                 5.00   
          Avg relative error                    0.00                 0.00   
          Avg time algorithm (ms)               0.14                 0.11   
          file size (MB)                       13.36                13.36   
          precomputing time (ms)             1016.00              1353.00   
8         Avg estimated dist                    5.00                 5.00   
          Avg relative error                    0.00                 0.00   
          Avg time algorithm (ms)               0.31                 0.23   
          file size (MB)                       26.72                26.72   
          precomputing time (ms)             1509.00              1791.00   
16        Avg estimated dist                    5.00                 5.00   
          Avg relative error                    0.00                 0.00   
          Avg time algorithm (ms)               0.96                 0.65   
          file size (MB)                       53.45                53.45   
          precomputing time (ms)             2993.00              3032.00   
32        Avg estimated dist                    5.00                 5.00   
          Avg relative error                    0.00                 0.00   
          Avg time algorithm (ms)               2.67                 1.90   
          file size (MB)                      106.90               106.90   
          precomputing time (ms)             4894.00              5138.00   
64        Avg estimated dist                    5.00                 5.00   
          Avg relative error                    0.00                 0.00   
          Avg time algorithm (ms)               9.46                 6.59   
          file size (MB)                      213.80               213.80   
          precomputing time (ms)             9313.00              9754.00   
128       Avg estimated dist                    5.00                 5.00   
          Avg relative error                    0.00                 0.00   
          Avg time algorithm (ms)              33.89                26.11   
          file size (MB)                      427.59               427.59   
          precomputing time (ms)            18113.00             18969.00   

                                   SelectBestCoverage  
landmarks metric                                       
4         Avg estimated dist                     5.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)                1.30  
          file size (MB)                        13.36  
          precomputing time (ms)             31183.00  
8         Avg estimated dist                     5.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)                2.50  
          file size (MB)                        26.72  
          precomputing time (ms)             31699.00  
16        Avg estimated dist                     5.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)                5.00  
          file size (MB)                        53.45  
          precomputing time (ms)             33086.00  
32        Avg estimated dist                     5.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)               10.99  
          file size (MB)                       106.90  
          precomputing time (ms)             34647.00  
64        Avg estimated dist                     5.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)               27.33  
          file size (MB)                       213.80  
          precomputing time (ms)             38506.00  
128       Avg estimated dist                     5.00  
  

In [85]:
import pandas as pd
import re
from collections import defaultdict

filepath = r'..\graph\landmarkAlgo\benchmark\output\web-NotreDame-landmarkBasic-bench.txt'

with open(filepath, 'r') as f:
    lines = f.readlines()

results = defaultdict(dict)

current_algorithm = None
current_landmarks = None

for line in lines:
    line = line.strip()

    if line.startswith('graph_theory/graph/landmarkAlgo'):
        current_algorithm = line.split('.')[-1]

    elif line.startswith('landmarks:'):
        current_landmarks = int(re.search(r'\d+', line).group())

    elif current_algorithm and current_landmarks:
        if line.startswith('precomputing time:'):
            results[(current_landmarks, 'precomputing time (ms)')][current_algorithm] = int(re.search(r'\d+', line).group())
        elif line.startswith('Avg estimated dist:'):
            results[(current_landmarks, 'Avg estimated dist')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())
        elif line.startswith('Avg relative error:'):
            results[(current_landmarks, 'Avg relative error')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())
        elif line.startswith('Avg time algorithm:'):
            results[(current_landmarks, 'Avg time algorithm (ms)')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())
        elif line.startswith('file size:'):
            results[(current_landmarks, 'file size (MB)')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())

df = pd.DataFrame(results).T
df.index.names = ['landmarks', 'metric']
df = df.sort_index()

df

SelectRandomNodes  SelectHighestDegree  \
landmarks metric                                                            
4         Avg estimated dist                 13.0000                 7.00   
          Avg relative error                  0.8571                 0.00   
          Avg time algorithm (ms)             0.0200                 0.02   
          file size (MB)                      1.2400                 1.24   
          precomputing time (ms)            216.0000               292.00   
8         Avg estimated dist                 11.0000                 7.00   
          Avg relative error                  0.5714                 0.00   
          Avg time algorithm (ms)             0.0400                 0.04   
          file size (MB)                      2.4900                 2.49   
          precomputing time (ms)            310.0000               387.00   
16        Avg estimated dist                 11.0000                 7.00   
          Avg relative error                  0.5714                 0.00   
          Avg time algorithm (ms)             0.0700                 0.08   
          file size (MB)                      4.9700                 4.97   
          precomputing time (ms)            575.0000               636.00   
32        Avg estimated dist                  9.0000                 7.00   
          Avg relative error                  0.2857                 0.00   
          Avg time algorithm (ms)             0.1700                 0.18   
          file size (MB)                      9.9400                 9.94   
          precomputing time (ms)           1016.0000              1101.00   
64        Avg estimated dist                 10.0000                 7.00   
          Avg relative error                  0.4286                 0.00   
          Avg time algorithm (ms)             0.4500                 0.48   
          file size (MB)                     19.8800                19.88   
          precomputing time (ms)           1926.0000              2069.00   
128       Avg estimated dist                  9.0000                 7.00   
          Avg relative error                  0.2857                 0.00   
          Avg time algorithm (ms)             1.4100                 1.48   
          file size (MB)                     39.7600                39.76   
          precomputing time (ms)           3759.0000              3870.00   

                                   SelectBestCoverage  
landmarks metric                                       
4         Avg estimated dist                     7.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)                0.18  
          file size (MB)                         1.24  
          precomputing time (ms)             10282.00  
8         Avg estimated dist                     7.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)                0.28  
          file size (MB)                         2.49  
          precomputing time (ms)             10198.00  
16        Avg estimated dist                     7.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)                0.50  
          file size (MB)                         4.97  
          precomputing time (ms)             10819.00  
32        Avg estimated dist                     7.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)                0.97  
          file size (MB)                         9.94  
          precomputing time (ms)             11488.00  
64        Avg estimated dist                     7.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)                3.48  
          file size (MB)                        19.88  
          precomputing time (ms)             22011.00  
128       Avg estimated dist                     7.00  
  

In [86]:
import pandas as pd
import re
from collections import defaultdict

filepath = r'..\graph\landmarkAlgo\benchmark\output\web-NotreDame-landmarkShortcut-bench.txt'

with open(filepath, 'r') as f:
    lines = f.readlines()

results = defaultdict(dict)

current_algorithm = None
current_landmarks = None

for line in lines:
    line = line.strip()

    if line.startswith('graph_theory/graph/landmarkAlgo'):
        current_algorithm = line.split('.')[-1]

    elif line.startswith('landmarks:'):
        current_landmarks = int(re.search(r'\d+', line).group())

    elif current_algorithm and current_landmarks:
        if line.startswith('precomputing time:'):
            results[(current_landmarks, 'precomputing time (ms)')][current_algorithm] = int(re.search(r'\d+', line).group())
        elif line.startswith('Avg estimated dist:'):
            results[(current_landmarks, 'Avg estimated dist')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())
        elif line.startswith('Avg relative error:'):
            results[(current_landmarks, 'Avg relative error')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())
        elif line.startswith('Avg time algorithm:'):
            results[(current_landmarks, 'Avg time algorithm (ms)')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())
        elif line.startswith('file size:'):
            results[(current_landmarks, 'file size (MB)')][current_algorithm] = float(re.search(r'\d+\.\d+', line).group())

df = pd.DataFrame(results).T
df.index.names = ['landmarks', 'metric']
df = df.sort_index()

df

SelectRandomNodes  SelectHighestDegree  \
landmarks metric                                                            
4         Avg estimated dist                    7.00                 7.00   
          Avg relative error                    0.00                 0.00   
          Avg time algorithm (ms)               0.13                 0.10   
          file size (MB)                        4.97                 4.97   
          precomputing time (ms)              296.00               447.00   
8         Avg estimated dist                    7.00                 7.00   
          Avg relative error                    0.00                 0.00   
          Avg time algorithm (ms)               0.26                 0.21   
          file size (MB)                        9.94                 9.94   
          precomputing time (ms)              433.00               480.00   
16        Avg estimated dist                    7.00                 7.00   
          Avg relative error                    0.00                 0.00   
          Avg time algorithm (ms)               0.57                 0.54   
          file size (MB)                       19.88                19.88   
          precomputing time (ms)              783.00               869.00   
32        Avg estimated dist                    7.00                 7.00   
          Avg relative error                    0.00                 0.00   
          Avg time algorithm (ms)               1.53                 1.58   
          file size (MB)                       39.76                39.76   
          precomputing time (ms)             1419.00              1878.00   
64        Avg estimated dist                    7.00                 7.00   
          Avg relative error                    0.00                 0.00   
          Avg time algorithm (ms)               4.84                 4.35   
          file size (MB)                       79.52                79.52   
          precomputing time (ms)             2935.00              2941.00   
128       Avg estimated dist                    7.00                 7.00   
          Avg relative error                    0.00                 0.00   
          Avg time algorithm (ms)              18.98                14.54   
          file size (MB)                      159.05               159.05   
          precomputing time (ms)             5960.00              6032.00   

                                   SelectBestCoverage  
landmarks metric                                       
4         Avg estimated dist                     7.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)                0.66  
          file size (MB)                         4.97  
          precomputing time (ms)             11034.00  
8         Avg estimated dist                     7.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)                1.34  
          file size (MB)                         9.94  
          precomputing time (ms)             11088.00  
16        Avg estimated dist                     7.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)                2.79  
          file size (MB)                        19.88  
          precomputing time (ms)             11985.00  
32        Avg estimated dist                     7.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)                6.19  
          file size (MB)                        39.76  
          precomputing time (ms)             11917.00  
64        Avg estimated dist                     7.00  
          Avg relative error                     0.00  
          Avg time algorithm (ms)               23.40  
          file size (MB)                        79.52  
          precomputing time (ms)             24656.00  
128       Avg estimated dist                     7.00  
  